# QuranHub — word-by-word audio pipeline (Quran.com → Bunny)

Builds the audio files for the `ar.wordbyword.hafs` edition:

1. Downloads every word recording from `audio.qurancdn.com/wbw/SSS_AAA_WWW.mp3` (77,422 files, ~3.7 GB).
2. Re-maps Quran.com word positions to **QuranHub word positions** (3 ayahs differ, see `source_name()`) and numbers each word with its **global word number** (1..77,432, the rank in surah → ayah → position order — the word-level analogue of the global ayah number used by the verse-by-verse files).
3. Converts each file to 128 kbps and 64 kbps mono MP3 with ffmpeg.
4. Uploads to Bunny storage under `quran/audio/wordbyword/{bitrate}/ar.wordbyword.hafs/{globalWordNumber}.mp3`.
5. Verifies every file through the CDN and writes a report.

**Before running**
- Add the Bunny storage access key as a Colab secret named `BUNNY_ACCESS_KEY` (🔑 icon in the left sidebar). The notebook never prints it.
- Optional: set `USE_DRIVE = True` to keep downloads/outputs on Google Drive so the run survives a disconnect.
- Expect roughly 3–4 hours on a free Colab runtime. Every step is resumable: re-run a cell and it skips finished files.


In [ ]:
# ---- Configuration ---------------------------------------------------------
EDITION_IDENTIFIER = "ar.wordbyword.hafs"
BITRATES = [128, 64]                     # must match edition.bitrates in the database
SOURCE_BASE = "https://audio.qurancdn.com/wbw"
STORAGE_HOST = "storage.bunnycdn.com"    # same host the existing merge_audio_files.py uses
STORAGE_ZONE = "quranhub-shared-public"
STORAGE_PREFIX = "quran/audio/wordbyword"
CDN_BASE = "https://quranhub.b-cdn.net/quran/audio/wordbyword"

USE_DRIVE = False                        # True = keep work dir on Google Drive (resumable across sessions)
WORK_DIR = "/content/wbw"

DOWNLOAD_THREADS = 24
CONVERT_WORKERS = 4                      # ffmpeg processes in parallel (Colab free tier has 2 vCPUs; 4 keeps them busy)
UPLOAD_THREADS = 16
VERIFY_THREADS = 32


In [ ]:
# ---- Secrets & environment ---------------------------------------------------
import os, getpass, subprocess, sys

try:
    from google.colab import userdata
    BUNNY_ACCESS_KEY = userdata.get("BUNNY_ACCESS_KEY")
except Exception:
    BUNNY_ACCESS_KEY = os.environ.get("BUNNY_ACCESS_KEY") or getpass.getpass("Bunny storage AccessKey: ")
assert BUNNY_ACCESS_KEY, "BUNNY_ACCESS_KEY is required"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/quranhub_wbw"

SRC_DIR = os.path.join(WORK_DIR, "src")
OUT_DIR = os.path.join(WORK_DIR, "out")
for d in [SRC_DIR] + [os.path.join(OUT_DIR, str(b)) for b in BITRATES]:
    os.makedirs(d, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "requests", "tqdm"], check=True)
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])
print("work dir:", WORK_DIR)


In [ ]:
# ---- QuranHub word table (words per ayah, exported from the database) --------
import json

# {surah: [words_in_ayah_1, words_in_ayah_2, ...]} — 114 surahs, 6236 ayahs, 77,432 words
EXPECTED_WORD_COUNTS = json.loads('''{"1":[4,4,2,3,4,3,9],"2":[1,7,8,12,8,11,12,11,10,12,11,7,19,16,7,11,17,6,19,25,11,23,20,13,34,39,20,13,19,28,15,12,22,13,18,19,11,17,10,13,18,8,7,10,8,8,11,19,18,10,12,8,7,26,14,7,18,20,18,28,59,24,15,13,12,9,20,23,18,17,24,10,11,37,19,23,9,11,24,23,13,10,29,15,49,13,28,10,26,27,30,11,27,17,9,25,18,12,10,10,22,74,12,12,25,19,17,18,33,17,18,17,31,30,12,14,11,24,10,31,15,11,18,19,22,30,14,17,16,18,9,16,27,15,15,31,19,10,14,30,15,21,45,34,32,14,7,19,17,31,17,6,10,13,12,10,9,24,20,11,13,9,9,43,31,12,23,16,11,22,18,14,25,29,11,14,51,37,8,16,14,16,14,32,44,17,65,17,27,13,25,6,15,22,14,73,29,26,12,24,14,8,25,17,14,12,11,15,12,17,20,20,49,30,22,25,57,16,26,26,39,27,16,14,14,13,7,40,46,30,45,33,64,25,47,25,35,8,15,27,6,7,28,9,16,54,43,30,60,15,27,9,52,21,50,24,24,43,67,39,24,24,11,38,27,35,29,13,16,15,20,28,30,17,45,11,19,13,17,14,15,128,32,28,27,49],"3":[1,7,11,18,11,13,46,15,13,16,14,9,29,24,24,11,6,18,26,26,18,11,20,15,15,25,21,27,22,26,14,11,11,7,18,24,34,15,20,17,20,12,7,20,20,7,23,5,45,19,8,20,9,6,30,13,11,7,15,7,25,15,6,31,15,19,14,13,13,8,10,17,32,8,36,9,27,29,28,13,33,7,14,29,13,18,9,9,11,14,24,15,24,11,11,10,25,12,19,14,17,12,34,14,14,17,10,11,11,23,11,35,14,15,9,18,23,30,27,21,11,12,10,13,15,17,9,12,16,12,5,5,11,13,23,15,13,6,9,24,6,13,12,27,24,21,19,10,12,6,19,40,25,75,21,36,14,7,30,18,21,13,8,25,20,9,33,16,13,19,11,15,17,15,11,22,12,17,36,30,22,9,28,12,23,24,21,20,9,11,21,11,20,14,43,7,7,21,30,10],"4":[28,16,28,14,16,37,19,13,15,13,71,88,18,13,22,14,20,24,34,17,11,17,54,42,58,14,13,8,23,13,12,27,19,40,22,32,15,18,15,15,12,14,48,14,9,38,28,20,14,10,20,11,10,18,11,20,20,25,28,30,15,16,16,23,19,27,6,3,18,7,10,17,18,21,31,20,48,36,18,12,24,13,30,22,22,15,17,22,28,36,32,60,16,39,33,8,27,12,10,31,23,62,20,22,15,7,14,21,18,13,11,14,33,26,20,20,11,8,21,7,7,21,20,16,17,12,40,29,22,11,32,10,12,14,37,28,20,6,14,33,36,19,16,17,11,18,11,14,14,22,8,17,37,19,22,6,34,8,14,14,14,26,25,14,15,13,11,12,11,23,53,18,26,11,15,50],"5":[23,52,61,31,43,61,19,26,10,7,23,45,32,25,23,18,42,30,28,22,14,17,21,16,13,14,24,16,13,9,25,40,34,13,13,24,12,13,14,19,67,26,15,41,31,26,16,51,33,10,22,28,15,37,13,11,23,12,20,27,16,13,13,47,12,24,24,30,20,19,19,33,25,8,25,16,23,17,10,19,15,26,21,16,14,7,17,12,45,15,20,13,24,23,50,20,27,9,11,15,25,9,21,24,19,51,26,22,15,63,13,21,15,22,17,44,29,11,24,11],"6":[14,14,12,11,12,31,16,13,9,13,10,25,9,24,9,9,17,7,37,13,14,12,12,10,32,11,16,16,9,19,26,13,14,22,28,10,19,22,17,15,12,11,14,20,9,23,14,15,8,28,17,25,15,27,6,21,22,14,31,22,17,12,17,10,27,9,5,23,12,45,39,8,24,14,9,15,18,17,12,25,22,11,14,21,7,8,9,16,17,16,44,22,49,29,17,12,16,13,44,15,18,16,9,15,8,13,13,25,21,13,22,23,12,23,11,18,11,10,30,12,20,24,15,29,28,9,10,35,8,27,10,9,18,7,18,31,20,26,22,20,31,16,23,40,39,28,13,37,8,27,42,35,17,17,8,14,35,37,19,16,16,9,8,29,21],"7":[1,13,14,10,12,6,6,9,11,11,16,17,13,5,4,7,15,12,16,25,5,28,12,12,7,19,30,22,16,16,15,26,29,11,18,11,37,40,14,23,10,15,36,28,10,18,13,14,15,20,19,10,38,32,8,15,29,17,20,9,10,11,13,14,15,14,10,7,27,17,25,13,34,25,23,8,15,6,14,13,11,13,7,8,37,24,19,21,42,12,6,13,15,13,18,18,9,9,10,19,23,9,16,8,18,11,6,6,9,7,7,4,11,5,10,11,11,6,4,3,4,3,19,8,5,16,21,16,22,10,22,12,13,22,11,11,29,23,10,9,18,23,41,14,20,36,13,22,18,39,11,15,14,15,41,30,43,31,8,43,20,17,27,19,18,11,19,14,40,10,19,26,15,5,13,30,9,10,30,14,7,8,5,11,22,10,37,28,29,12,7,7,12,14,26,9,10,12,7,10,12,7,23,8,16,11],"8":[18,17,6,11,11,12,23,7,11,18,22,23,13,6,11,20,21,6,24,11,8,10,11,20,13,19,11,10,18,16,17,19,12,21,13,21,18,16,16,10,33,35,20,19,12,13,17,36,18,13,9,17,18,17,10,12,10,14,8,33,12,12,20,8,26,26,20,10,11,24,13,45,13,18,22],"9":[9,14,32,22,26,18,23,16,13,10,13,18,20,11,11,24,18,23,24,16,10,8,19,33,23,18,12,27,29,23,23,15,14,28,19,36,31,28,16,43,15,23,13,14,14,15,17,16,15,16,14,22,12,22,17,10,11,16,19,22,26,12,16,17,13,16,19,16,32,26,24,24,11,46,12,9,15,11,20,25,27,8,25,18,16,18,11,14,13,17,26,23,19,30,18,14,16,16,27,25,22,17,16,16,16,12,24,23,27,15,37,16,20,22,18,17,27,31,8,52,18,22,16,18,12,16,21,14,15],"10":[5,26,29,28,23,14,15,6,15,14,18,27,17,10,39,18,14,29,17,16,22,41,25,43,11,16,27,18,10,16,27,11,10,17,29,17,22,16,19,12,15,11,11,10,20,15,12,7,22,12,11,13,11,21,16,5,14,10,20,19,42,10,4,15,10,25,15,25,9,13,33,16,17,22,14,10,11,16,6,10,17,7,22,12,10,5,16,29,11,27,7,14,24,22,10,8,8,21,15,14,14,15,10,26,9,16,24,22,11],"11":[10,9,25,8,19,16,31,22,11,13,10,27,18,16,14,16,39,23,11,24,9,7,14,11,9,11,29,19,22,9,31,13,10,18,12,17,11,19,10,26,11,19,23,17,15,23,19,18,19,17,13,17,15,16,7,18,21,14,12,17,27,21,22,17,11,18,8,12,15,18,9,12,14,11,5,14,13,26,13,11,29,12,8,26,14,11,20,35,22,9,18,15,19,18,10,6,10,9,9,9,24,12,16,5,10,9,15,18,20,18,10,12,17,14,7,26,8,10,15,17,9,3,16],"12":[5,6,16,15,15,25,7,15,14,16,11,8,13,9,17,4,18,18,16,9,32,9,23,19,23,19,10,13,10,18,32,19,17,10,11,27,26,26,9,32,19,18,24,8,11,22,14,15,11,22,29,12,15,14,8,17,7,9,17,10,6,14,16,17,25,24,27,30,16,15,5,12,12,6,11,33,24,15,13,37,17,11,18,12,12,13,20,21,10,23,9,11,12,12,6,20,8,9,14,46,21,14,6,10,11,7,14,17,29,19,23],"13":[15,26,24,27,25,20,15,15,5,14,35,9,19,27,11,45,42,27,15,7,13,19,15,7,23,16,19,11,8,28,51,13,39,14,19,25,20,21,8,13,16,18,15],"14":[16,14,16,19,21,26,11,13,37,33,29,17,16,10,5,7,16,22,14,5,34,44,17,15,12,13,18,12,4,12,21,25,9,15,13,15,26,18,13,9,8,13,9,26,14,12,11,10,6,6,10,13],"15":[6,7,7,8,7,8,8,9,7,7,8,5,7,9,8,7,5,7,11,8,11,12,6,7,7,8,7,12,9,4,7,8,11,5,6,6,4,4,10,4,5,11,4,8,5,3,10,8,6,5,4,9,7,8,7,8,5,6,6,6,5,4,7,4,14,9,4,6,4,5,6,5,3,8,5,3,5,5,5,5,5,6,3,6,14,5,7,15,5,4,4,3,3,6,3,8,7,6,5],"16":[9,18,7,8,8,7,14,9,10,13,16,15,13,21,11,4,7,10,6,10,7,11,13,9,15,21,21,19,8,20,15,12,22,10,32,28,14,18,11,10,18,5,15,12,16,7,8,17,14,7,12,11,12,10,6,12,7,9,19,12,23,17,17,14,16,17,15,13,24,20,22,21,15,10,27,29,19,15,18,25,25,6,7,14,11,19,10,13,24,16,20,32,17,19,15,15,18,8,10,9,17,13,16,12,11,24,13,11,7,18,15,24,9,13,22,22,5,15,19,11,7,9,12,17,22,12,14,8],"17":[21,12,9,13,17,11,23,11,16,9,8,23,13,7,21,14,13,18,12,12,11,9,25,12,12,10,9,12,13,12,13,8,22,17,11,16,13,7,18,11,10,13,6,21,12,19,19,9,9,5,25,9,16,14,15,13,18,18,21,25,13,15,10,19,9,14,18,15,22,17,15,11,13,9,12,13,11,14,12,14,9,13,12,11,14,13,9,19,14,10,11,12,26,15,14,11,27,14,23,14,18,15,9,14,9,9,18,8,5,22,21],"18":[11,15,3,6,16,11,11,6,10,16,7,9,11,19,18,19,34,22,38,13,32,33,7,17,8,23,15,30,28,12,28,14,11,13,12,12,16,8,19,15,8,19,11,9,23,13,11,16,27,26,13,13,10,14,19,18,30,19,8,13,11,12,20,9,12,11,6,8,10,12,15,8,10,16,9,13,24,12,17,10,9,35,9,10,2,24,13,13,3,15,6,3,13,19,12,19,8,15,11,5,11,14,5,11,14,9,10,6,18,24],"19":[1,5,5,14,13,8,12,14,14,13,11,7,6,6,9,10,11,8,9,11,15,5,13,10,8,18,10,11,10,8,11,6,9,9,16,8,11,11,12,8,8,15,13,9,11,12,10,14,14,9,10,7,7,11,9,8,3,29,11,12,11,11,9,18,12,8,10,8,10,8,9,8,16,9,25,13,8,7,9,5,8,6,9,7,6,5,9,4,4,9,4,6,10,4,5,9,10,14],"20":[1,5,4,6,4,12,7,8,4,18,4,9,5,11,10,10,4,13,3,5,7,11,4,5,5,3,4,2,5,2,3,3,3,2,4,5,5,6,22,34,2,8,5,8,10,7,21,10,4,9,5,11,19,9,8,6,7,14,8,6,15,5,13,10,11,12,5,6,16,8,26,19,14,13,10,12,18,8,5,14,17,9,5,9,9,26,15,12,12,16,9,7,4,18,4,16,28,12,13,8,7,8,6,12,7,3,6,13,12,10,9,11,14,17,11,9,12,7,6,13,15,6,18,12,8,8,12,17,9,20,18,12,13,19,11],"21":[7,11,15,10,14,8,14,9,8,8,10,7,10,5,8,7,11,13,13,5,7,13,6,22,15,8,6,15,13,18,13,8,11,10,10,17,8,7,16,10,13,13,14,18,11,11,20,8,8,7,9,11,5,8,7,13,7,8,8,7,8,6,9,7,9,11,9,7,7,5,8,8,15,16,6,12,12,14,14,10,14,12,10,16,7,6,23,8,11,19,11,8,6,12,7,10,19,11,9,7,9,9,11,16,12,6,5,11,13,8,8,10],"22":[9,22,12,11,69,13,12,13,15,9,25,14,10,17,22,9,22,37,17,6,4,12,22,9,25,16,14,20,8,24,21,9,11,20,15,27,20,13,10,35,17,9,4,11,14,23,14,11,8,8,8,27,16,21,16,12,8,18,7,16,15,17,15,12,27,10,17,7,9,18,18,29,28,9,11,10,11,44],"23":[3,5,5,4,4,10,7,5,5,3,6,7,6,21,5,5,10,13,13,9,15,4,16,26,10,5,32,15,8,7,6,14,25,7,9,4,10,12,5,5,8,6,7,19,8,7,7,4,6,11,11,8,9,5,7,7,7,5,5,10,7,12,14,8,7,9,4,10,7,10,16,9,5,8,11,8,12,10,7,10,6,8,12,8,5,8,5,14,5,5,24,6,6,6,7,9,7,5,8,16,10,6,10,6,8,8,7,5,14,8,8,7,8,9,8,11,18,7],"24":[9,26,18,19,11,17,9,11,9,9,28,12,14,15,16,14,9,7,20,9,34,26,13,9,11,16,18,22,17,15,78,17,48,13,48,14,19,14,22,33,21,7,38,10,31,11,15,11,7,16,18,10,17,23,38,8,11,49,19,23,76,39,27,22],"25":[9,19,22,16,9,12,16,17,9,17,8,9,9,8,13,10,16,21,15,20,19,10,9,7,7,9,11,6,11,9,11,15,8,11,9,9,13,8,7,16,11,17,9,15,16,5,11,13,10,9,7,7,15,12,14,5,14,13,17,13,11,13,12,5,11,4,11,22,8,16,9,9,10,13,9,5,12],"26":[1,4,6,11,11,8,11,8,5,8,4,6,8,6,7,7,5,10,7,6,11,8,5,9,5,5,7,9,8,5,7,6,6,7,8,7,4,5,5,7,12,6,7,9,8,3,4,3,21,7,11,8,5,4,3,3,4,3,4,2,8,6,13,3,5,3,8,5,4,6,6,5,4,6,5,3,6,4,4,4,4,8,6,6,5,6,4,6,6,3,3,6,7,4,3,4,6,4,4,4,3,7,8,5,4,7,4,3,11,3,5,6,7,4,5,8,5,9,6,4,8,5,3,7,4,3,11,5,4,4,3,5,3,2,6,9,5,3,10,5,3,7,4,3,11,5,3,4,5,3,4,6,5,11,9,7,3,10,5,4,7,4,3,11,4,11,8,5,5,3,4,3,6,8,5,4,6,4,3,11,6,3,9,5,5,9,9,5,10,8,5,4,4,5,3,4,9,5,6,5,7,5,4,2,4,5,6,7,4,4,5,4,9,3,6,7,4,4,3,4,6,5,4,3,7,5,19],"27":[6,3,9,10,10,7,16,14,6,19,11,19,8,11,15,18,9,19,24,12,9,14,11,17,14,8,7,11,8,8,5,12,12,13,8,16,13,10,16,38,12,14,12,28,13,12,13,10,14,7,9,11,5,8,11,15,7,6,12,27,21,16,20,19,14,14,9,12,9,9,7,9,10,8,10,12,4,8,7,11,14,16,11,13,8,15,18,18,11,13,17,15,11],"28":[1,4,9,19,12,12,22,13,18,17,10,16,17,10,38,13,9,16,32,18,10,11,27,15,26,10,29,15,27,19,19,25,9,14,15,17,18,27,12,10,9,10,16,13,18,20,18,26,13,23,6,8,15,12,15,13,25,17,20,15,17,8,18,13,6,7,11,14,7,15,19,21,13,8,17,29,26,28,20,16,17,26,14,18,19,15,16,19],"29":[1,10,11,10,12,10,12,21,7,31,6,18,11,16,6,13,27,13,13,19,8,17,13,20,27,12,18,14,23,6,15,16,23,12,7,16,7,16,13,28,19,12,8,10,21,25,19,13,14,16,15,19,12,6,13,8,7,17,5,12,13,14,22,15,16,6,15,17,9],"30":[1,2,8,12,8,11,10,22,32,13,8,5,9,5,9,11,6,8,14,11,20,13,14,21,16,8,18,27,16,22,9,11,18,6,10,15,15,16,22,22,15,14,16,9,11,17,18,28,10,19,9,11,14,24,12,20,9,19,8,10],"31":[1,4,3,9,8,18,16,8,8,27,14,18,14,17,29,24,17,16,11,30,21,15,15,7,15,10,21,11,25,16,19,22,30,27],"32":[1,8,18,24,17,6,10,8,14,14,11,16,16,14,15,11,13,8,12,21,9,13,14,10,11,18,17,7,11,5],"33":[13,12,6,28,28,29,17,8,21,16,6,13,25,14,13,15,24,14,35,23,17,18,18,16,16,16,13,15,14,16,14,19,25,14,30,24,48,22,13,17,7,3,13,8,7,6,8,12,22,61,35,25,69,11,30,13,14,12,21,19,6,12,15,7,8,11,8,8,16,8,14,18,15],"34":[17,18,32,10,11,16,16,16,29,12,12,27,19,27,20,16,8,17,19,10,21,26,22,17,9,11,12,12,7,10,32,15,34,14,8,13,23,9,20,10,13,18,34,11,14,24,17,7,8,16,10,9,10,15],"35":[24,19,21,11,14,13,13,25,17,24,34,28,28,18,10,6,5,34,4,4,4,16,4,12,13,7,22,17,16,8,16,21,13,11,14,18,25,10,24,33,19,19,26,30,25],"36":[1,2,3,3,3,7,8,10,12,8,12,14,8,11,15,6,5,12,9,10,7,8,15,5,4,7,7,13,8,11,11,6,10,10,8,13,9,8,7,15,8,6,9,6,11,11,24,7,8,7,10,12,10,11,7,7,6,5,4,13,5,8,5,5,11,9,10,7,12,8,12,6,6,7,7,9,11,11,10,11,14,10,8],"37":[2,2,2,3,7,6,5,9,4,7,12,3,4,4,6,7,2,4,7,5,7,7,7,3,4,4,5,6,5,10,6,4,5,4,10,6,5,4,6,4,4,3,3,3,5,3,7,4,3,5,7,4,7,4,5,5,6,3,6,5,4,6,4,6,4,6,7,5,4,4,5,4,5,4,5,5,4,4,5,4,4,3,4,5,6,5,4,4,3,3,6,4,4,3,4,4,7,5,6,5,3,26,4,3,7,5,3,4,3,3,4,5,10,5,5,4,3,3,4,4,4,4,4,5,5,5,3,4,4,3,4,4,4,4,4,3,4,3,4,5,4,4,5,6,4,5,6,4,5,6,5,4,4,4,2,4,5,10,4,4,3,4,5,6,3,3,3,6,4,4,5,3,4,4,3,2,6,4,3,6,3,4],"38":[4,6,10,10,8,12,10,15,7,10,6,8,7,7,10,8,11,7,5,6,7,23,15,32,9,30,16,13,9,7,6,11,6,9,15,8,4,4,7,6,11,6,10,14,8,5,5,8,6,5,7,4,5,7,5,4,4,4,10,11,11,10,6,6,11,7,4,3,9,8,9,9,4,6,14,10,5,6,6,4,4,4,4,4,7,10,5,4],"39":[6,10,33,16,23,36,31,34,24,21,9,5,9,6,19,16,12,13,10,20,31,20,34,13,10,12,11,7,19,4,7,15,8,8,12,15,12,36,9,8,18,26,14,11,18,17,25,11,22,11,15,15,20,13,16,14,9,12,10,14,11,9,11,7,14,6,17,22,15,9,34,9,20,16,17],"40":[1,6,14,13,21,10,27,16,12,15,13,15,14,8,16,15,13,15,6,15,30,13,6,7,19,18,13,39,26,10,14,6,16,30,23,9,23,8,11,23,9,15,22,12,10,13,19,12,13,16,11,10,8,4,12,23,12,13,11,13,20,12,7,21,14,20,30,12,10,9,6,6,7,17,11,8,14,32,9,12,6,25,15,12,17],"41":[1,4,7,7,17,16,8,9,14,15,16,19,9,23,25,19,14,5,8,12,18,20,9,11,25,11,10,14,16,19,16,4,13,17,11,11,19,12,22,26,9,13,17,30,18,11,28,12,11,32,13,17,19,12],"42":[1,1,10,10,19,12,22,19,16,15,19,13,38,30,34,18,11,22,9,22,22,24,29,22,12,12,17,13,16,10,14,6,15,7,9,17,10,11,6,14,9,15,8,20,27,16,21,27,15,11,22,27,15],"43":[1,2,6,7,8,6,8,7,10,11,12,11,20,4,9,7,12,9,12,15,8,11,21,14,7,9,5,7,9,9,10,25,17,5,12,11,7,11,9,11,6,7,8,6,13,12,7,13,11,7,17,11,11,7,6,4,9,14,10,8,10,7,17,9,11,10,7,8,5,5,15,7,6,6,6,6,8,7,5,11,8,8,8,11,13,13,8,7,6],"44":[1,2,8,5,6,7,8,10,5,6,5,6,7,6,6,6,8,9,9,6,5,6,5,6,5,3,4,4,8,7,7,6,7,3,8,5,12,7,8,5,10,8,3,2,4,2,5,7,5,6,5,3,5,4,5,10,7,5,3],"45":[1,6,6,9,20,12,4,14,11,19,11,14,16,14,11,13,24,13,15,7,17,13,24,21,16,18,10,14,11,13,12,22,11,14,16,8,8],"46":[1,6,15,29,18,9,14,25,24,24,19,16,13,9,45,18,28,17,9,27,25,12,13,18,14,32,10,17,18,19,14,18,22,18,27],"47":[8,18,18,38,3,4,9,6,8,16,11,22,14,13,44,24,6,15,14,27,12,10,7,7,16,15,7,10,11,12,8,21,10,15,12,13,6,29],"48":[5,15,4,19,19,21,8,5,8,25,37,20,9,14,31,28,27,19,7,18,14,12,12,19,40,28,28,14,54],"49":[15,22,17,9,12,17,28,7,30,10,39,31,20,27,18,16,19,10],"50":[3,11,7,9,9,12,11,5,10,5,8,7,4,9,10,14,8,8,9,6,6,12,6,6,4,10,10,8,8,9,5,6,7,5,6,15,13,13,12,5,7,7,6,9,13],"51":[2,2,2,2,3,3,3,4,4,2,5,4,5,7,5,9,6,3,5,4,4,5,9,6,9,6,5,9,9,8,5,6,5,4,6,7,7,8,6,7,7,9,8,8,7,8,5,4,7,8,11,13,6,5,5,6,9,7,9,7],"52":[1,2,3,2,2,2,4,4,4,3,3,5,6,6,6,12,5,8,6,7,19,5,8,7,5,7,6,9,8,7,6,8,6,6,8,7,7,9,5,7,5,7,9,9,7,9,10,10,5],"53":[3,5,4,5,3,3,3,3,5,5,5,4,4,3,3,5,5,6,3,3,4,4,25,4,3,18,9,16,11,16,16,31,3,3,5,7,3,5,6,4,4,4,4,4,5,4,4,4,4,4,3,9,2,3,4,5,2,6,4,3,2,3],"54":[4,7,6,7,5,8,8,8,9,5,5,9,5,6,6,4,7,6,9,6,4,7,3,10,9,5,7,8,4,4,8,7,4,9,7,5,9,5,3,7,5,7,9,5,4,6,5,9,5,6,6,5,4,5,6],"55":[1,2,2,2,3,3,4,4,6,3,5,4,4,5,6,4,4,4,3,4,4,4,4,6,4,4,6,4,10,4,4,4,16,4,8,4,6,4,8,4,6,4,6,5,4,5,4,2,4,3,4,5,4,9,4,9,4,3,4,5,4,3,4,1,4,3,4,4,4,3,4,4,4,6,4,6,4,6],"56":[3,3,2,4,3,3,3,5,5,2,2,3,3,3,3,3,4,5,5,3,4,2,3,4,6,4,5,3,2,2,2,2,4,2,3,2,2,2,3,3,5,3,3,4,5,5,9,2,4,5,5,5,3,4,3,4,4,3,5,7,9,6,3,5,6,2,3,4,7,6,4,6,5,4,4,5,3,3,4,4,4,4,4,3,7,5,4,5,4,6,5,6,3,2,5,4],"57":[9,11,9,36,8,12,15,15,17,37,12,22,27,20,15,28,13,12,20,39,25,21,15,12,28,14,39,21,22],"58":[18,23,21,26,18,15,47,38,19,17,30,22,24,19,10,10,15,17,15,8,9,51],"59":[11,40,14,12,13,25,37,18,31,23,28,16,12,23,11,17,10,17,10,10,18,13,19,17],"60":[48,13,13,52,13,19,15,21,21,55,21,38,19],"61":[11,8,9,11,23,31,16,11,14,11,15,17,9,35],"62":[12,21,8,10,23,17,9,18,21,15,21],"63":[17,12,11,22,14,18,20,17,17,21,11],"64":[17,11,9,14,13,16,18,10,27,11,16,11,9,18,8,16,12,5],"65":[41,30,21,28,14,32,25,14,7,16,34,25],"66":[14,11,29,21,17,22,11,45,11,27,24,17],"67":[9,11,18,10,12,7,8,13,18,11,5,9,9,7,14,11,12,8,15,15,12,12,12,8,7,9,13,15,13,10],"68":[4,5,5,4,2,2,11,3,4,5,3,4,4,5,7,3,10,2,7,2,2,7,3,6,4,5,3,7,6,5,5,10,8,6,3,4,5,5,12,4,8,9,11,10,5,7,5,11,10,5,12,5],"69":[1,2,4,4,4,6,15,5,6,6,7,6,6,6,3,5,9,6,9,5,4,3,2,8,10,4,3,4,3,2,3,7,6,5,5,5,4,4,3,4,7,6,4,5,3,4,6,3,5,4,3,4],"70":[4,4,4,11,3,3,2,4,3,4,9,2,3,6,3,2,4,2,4,4,4,2,5,5,2,4,6,5,4,10,7,5,4,5,4,5,5,8,5,7,8,8,9,9],"71":[14,6,5,18,7,5,14,4,7,6,4,9,6,3,8,7,5,5,5,4,13,3,13,8,12,10,10,15],"72":[13,9,9,7,10,11,9,8,13,13,9,11,14,10,5,7,10,8,10,8,8,12,16,11,11,7,13,13],"73":[2,4,5,6,5,8,6,6,9,7,6,4,5,8,11,6,8,6,9,78],"74":[2,2,2,2,2,3,2,4,4,4,4,4,2,3,4,5,2,3,3,4,2,3,3,6,5,2,4,4,2,3,57,2,3,3,3,2,7,5,3,3,2,4,5,4,4,4,3,4,5,3,3,9,5,3,3,11],"75":[4,4,5,6,5,4,3,2,3,5,3,4,6,5,3,6,4,4,4,4,2,3,3,3,5,4,3,3,3,4,4,3,5,3,4,5,6,5,5,7],"76":[11,10,7,6,8,7,7,7,10,7,8,5,10,6,8,5,6,4,9,7,13,8,6,9,5,7,8,9,9,11,10],"77":[2,2,2,2,2,3,3,3,3,3,3,3,2,5,3,3,3,3,3,5,4,3,3,3,4,2,7,3,6,6,6,4,3,3,4,4,3,5,5,3,5,3,6,4,3,5,3,6,3,4],"78":[2,3,4,2,3,4,2,2,3,3,3,4,3,5,4,2,5,6,4,4,4,2,3,6,3,2,5,3,4,5,3,2,2,2,6,5,10,14,9,15],"79":[2,2,2,2,2,3,2,3,2,5,4,5,4,3,4,6,5,6,4,3,2,3,2,4,5,6,6,3,4,4,4,2,3,4,5,4,3,3,4,9,4,5,4,3,5,9],"80":[2,3,4,4,3,3,4,4,2,3,3,3,3,2,2,2,4,4,4,3,3,4,5,4,4,4,3,2,2,2,2,3,3,5,2,2,6,3,2,4,2,4],"81":[3,3,3,3,3,3,3,3,3,3,3,3,3,4,3,2,3,3,4,6,3,3,4,5,5,2,5,5,8],"82":[3,3,3,3,5,6,4,6,4,3,2,3,4,4,3,4,5,6,9],"83":[2,6,5,5,2,5,6,4,2,3,4,7,7,8,6,4,7,6,4,2,2,4,3,5,4,6,3,4,8,4,6,6,4,6,3,6],"84":[3,3,3,4,3,8,5,4,4,6,3,2,5,5,6,3,3,3,4,4,6,4,4,3,9],"85":[3,2,2,3,3,4,6,9,10,14,14,4,4,3,3,3,4,2,5,4,4,3],"86":[2,4,2,6,4,4,5,4,3,6,3,3,3,3,3,2,4],"87":[4,3,3,3,3,3,9,2,4,3,2,4,6,4,4,4,3,5,3],"88":[4,3,2,3,4,6,6,3,2,3,4,3,3,2,2,2,6,4,4,4,4,3,4,4,3,4],"89":[1,2,2,3,6,6,3,6,5,3,4,3,5,3,11,10,5,5,4,4,6,5,9,4,5,4,3,5,3,2],"90":[4,4,3,5,6,4,5,4,2,2,3,4,2,6,3,4,9,3,6,3],"91":[2,3,3,3,3,3,3,3,4,4,3,3,7,7,3],"92":[3,3,4,3,4,2,2,4,2,2,6,3,4,3,4,3,2,4,6,5,2],"93":[1,3,5,5,4,4,3,3,4,4,4],"94":[4,3,3,3,4,4,3,3],"95":[2,2,3,6,4,9,4,4],"96":[5,4,3,3,5,4,3,4,3,3,5,3,4,5,6,3,2,2,5],"97":[5,5,6,9,5],"98":[12,6,3,11,16,16,9,21],"99":[4,3,4,3,4,6,6,6],"100":[2,2,2,3,3,4,4,4,7,4,5],"101":[1,2,4,5,4,4,4,4,2,4,2],"102":[2,3,3,4,5,2,4,5],"103":[1,4,9],"104":[4,4,4,4,4,3,4,3,3],"105":[7,5,4,4,3],"106":[2,4,4,7],"107":[4,4,5,2,5,3,2],"108":[3,3,4],"109":[3,4,5,5,5,4],"110":[5,7,7],"111":[5,6,4,3,5],"112":[4,2,4,5],"113":[4,4,5,5,5],"114":[4,2,2,4,5,3]}''')

# Quran.com has no recording for these QuranHub word locations (verified 2026-10-12).
MISSING_AUDIO = {"8:66:2", "12:8:15", "18:49:20", "18:54:7", "60:2:7", "79:1:2", "98:7:1"}

# Quran.com treats "بَعْدَ مَا" as ONE word in these ayahs; QuranHub has it as two words.
# (surah, ayah) -> QuranHub position of "بَعْدَ". Both halves get the merged clip, later words shift by -1.
MERGED_BAADA_MA = {(2, 181): 3, (8, 6): 4, (13, 37): 8}


def source_name(surah, ayah, position):
    """Quran.com file stem (SSS_AAA_WWW) for a QuranHub word position."""
    merged_at = MERGED_BAADA_MA.get((surah, ayah))
    src_pos = position
    if merged_at is not None:
        if position == merged_at + 1:
            src_pos = merged_at            # "مَا" plays the merged "بَعْدَ مَا" clip
        elif position > merged_at + 1:
            src_pos = position - 1
    return f"{surah:03d}_{ayah:03d}_{src_pos:03d}"


# Global word number = 1-based rank in surah -> ayah -> position order (1..77432), the word-level
# analogue of the global ayah number. The API computes the same ranks from the word table.
TARGETS = []  # (surah, ayah, position, global_word_number, source_stem)
word_number = 0
for surah in range(1, 115):
    for ayah_idx, n_words in enumerate(EXPECTED_WORD_COUNTS[str(surah)], start=1):
        for position in range(1, n_words + 1):
            word_number += 1
            if f"{surah}:{ayah_idx}:{position}" in MISSING_AUDIO:
                continue
            TARGETS.append((surah, ayah_idx, position, word_number, source_name(surah, ayah_idx, position)))

assert len(EXPECTED_WORD_COUNTS) == 114 and sum(len(v) for v in EXPECTED_WORD_COUNTS.values()) == 6236
assert word_number == 77432, word_number
assert len(TARGETS) == 77432 - len(MISSING_AUDIO) == 77425, len(TARGETS)
BY_LOCATION = {(s, a, p): n for s, a, p, n, _ in TARGETS}
# numbering self-checks (ranks verified against the database)
assert BY_LOCATION[(1, 1, 1)] == 1 and BY_LOCATION[(1, 7, 9)] == 29 and BY_LOCATION[(2, 1, 1)] == 30
assert BY_LOCATION[(2, 181, 3)] == 3289 and BY_LOCATION[(2, 282, 128)] == 6010
assert BY_LOCATION[(37, 130, 3)] == 58041 and BY_LOCATION[(114, 6, 3)] == 77432
# mapping self-checks
assert source_name(2, 181, 3) == "002_181_003" and source_name(2, 181, 4) == "002_181_003"
assert source_name(2, 181, 14) == "002_181_013" and source_name(8, 6, 12) == "008_006_011"
assert source_name(13, 37, 20) == "013_037_019" and source_name(1, 1, 4) == "001_001_004"
SOURCE_STEMS = sorted({t[4] for t in TARGETS})
print("targets:", len(TARGETS), "distinct source files:", len(SOURCE_STEMS))


In [ ]:
# ---- Step 1: download source files (resumable) --------------------------------
import requests, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

session = requests.Session()
session.headers["User-Agent"] = "quranhub-wbw-pipeline/1.0"


def download(stem):
    dest = os.path.join(SRC_DIR, stem + ".mp3")
    if os.path.exists(dest) and os.path.getsize(dest) > 1000:
        return stem, "cached"
    url = f"{SOURCE_BASE}/{stem}.mp3"
    for attempt in range(5):
        try:
            r = session.get(url, timeout=60)
            if r.status_code == 404:
                return stem, "404"
            r.raise_for_status()
            if not r.content.startswith((b"ID3", b"\xff\xfb", b"\xff\xf3", b"\xff\xf2")):
                raise ValueError("not an mp3")
            tmp = dest + ".part"
            with open(tmp, "wb") as f:
                f.write(r.content)
            os.replace(tmp, dest)
            return stem, "ok"
        except Exception as e:
            err = str(e)
            time.sleep(1.5 * (attempt + 1))
    return stem, "error: " + err


download_status = {}
with ThreadPoolExecutor(DOWNLOAD_THREADS) as ex:
    futures = [ex.submit(download, s) for s in SOURCE_STEMS]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="download"):
        stem, status = fut.result()
        download_status[stem] = status

from collections import Counter
print(Counter("ok/cached" if v in ("ok", "cached") else v for v in download_status.values()))
failed_downloads = sorted(k for k, v in download_status.items() if v not in ("ok", "cached"))
print("failed:", failed_downloads[:20], "..." if len(failed_downloads) > 20 else "")
assert not failed_downloads, "Re-run this cell until there are no failed downloads"


In [ ]:
# ---- Step 2: convert to mono MP3 at each bitrate (resumable) --------------------
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed


def out_path(bitrate, word_number):
    return os.path.join(OUT_DIR, str(bitrate), f"{word_number}.mp3")


def convert(target):
    surah, ayah, position, word_number, stem = target
    src = os.path.join(SRC_DIR, stem + ".mp3")
    results = []
    for bitrate in BITRATES:
        dest = out_path(bitrate, word_number)
        if os.path.exists(dest) and os.path.getsize(dest) > 500:
            results.append("cached")
            continue
        tmp = dest + ".part.mp3"
        cmd = ["ffmpeg", "-nostdin", "-loglevel", "error", "-y", "-i", src,
               "-vn", "-ac", "1", "-ar", "44100", "-codec:a", "libmp3lame", "-b:a", f"{bitrate}k",
               "-map_metadata", "-1", tmp]
        proc = subprocess.run(cmd, capture_output=True, text=True)
        if proc.returncode != 0 or not os.path.exists(tmp) or os.path.getsize(tmp) < 500:
            results.append("error: " + proc.stderr.strip()[:200])
            continue
        os.replace(tmp, dest)
        results.append("ok")
    return target, results


convert_errors = []
with ThreadPoolExecutor(CONVERT_WORKERS) as ex:
    futures = [ex.submit(convert, t) for t in TARGETS]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="convert"):
        target, results = fut.result()
        for r in results:
            if r.startswith("error"):
                convert_errors.append((target, r))

print("conversion errors:", len(convert_errors))
for item in convert_errors[:10]:
    print("  ", item)
assert not convert_errors, "Re-run this cell until there are no conversion errors"
for bitrate in BITRATES:
    n = len([f for f in os.listdir(os.path.join(OUT_DIR, str(bitrate))) if f.endswith(".mp3") and not f.endswith(".part.mp3")])
    print(f"{bitrate} kbps: {n} files (expected {len(TARGETS)})")


In [ ]:
# ---- Step 3: upload to Bunny storage (resumable via manifest) ------------------
import json, time
from concurrent.futures import ThreadPoolExecutor, as_completed

MANIFEST = os.path.join(WORK_DIR, "uploaded.json")
uploaded = set(json.load(open(MANIFEST))) if os.path.exists(MANIFEST) else set()


def storage_path(bitrate, word_number):
    return f"{STORAGE_PREFIX}/{bitrate}/{EDITION_IDENTIFIER}/{word_number}.mp3"


def upload(job):
    bitrate, word_number = job
    key = storage_path(bitrate, word_number)
    if key in uploaded:
        return key, "cached"
    local = out_path(bitrate, word_number)
    url = f"https://{STORAGE_HOST}/{STORAGE_ZONE}/{key}"
    headers = {"AccessKey": BUNNY_ACCESS_KEY, "Content-Type": "audio/mpeg"}
    with open(local, "rb") as f:
        data = f.read()
    for attempt in range(5):
        try:
            r = requests.put(url, data=data, headers=headers, timeout=120)
            if r.status_code in (200, 201):
                return key, "ok"
            if r.status_code in (401, 403):
                return key, f"auth error {r.status_code}"
            last = f"http {r.status_code}"
        except Exception as e:
            last = str(e)
        time.sleep(2 * (attempt + 1))
    return key, "error: " + last


jobs = [(b, n) for b in BITRATES for (_, _, _, n, _) in TARGETS]
upload_errors = []
done_since_save = 0
with ThreadPoolExecutor(UPLOAD_THREADS) as ex:
    futures = [ex.submit(upload, j) for j in jobs]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="upload"):
        key, status = fut.result()
        if status in ("ok", "cached"):
            uploaded.add(key)
            done_since_save += 1
            if done_since_save >= 2000:
                json.dump(sorted(uploaded), open(MANIFEST, "w"))
                done_since_save = 0
        else:
            upload_errors.append((key, status))
            if status.startswith("auth error"):
                break

json.dump(sorted(uploaded), open(MANIFEST, "w"))
print("uploaded:", len(uploaded), "of", len(jobs), "| errors:", len(upload_errors))
for item in upload_errors[:10]:
    print("  ", item)
assert not upload_errors, "Re-run this cell until there are no upload errors"


In [ ]:
# ---- Step 4: verify every file through the CDN and write a report -------------
from concurrent.futures import ThreadPoolExecutor, as_completed


def head(job):
    bitrate, word_number = job
    url = f"{CDN_BASE}/{bitrate}/{EDITION_IDENTIFIER}/{word_number}.mp3"
    for attempt in range(4):
        try:
            r = requests.head(url, timeout=30)
            if r.status_code == 200 and int(r.headers.get("Content-Length") or 0) > 500:
                return url, "ok"
            if r.status_code == 404:
                return url, "404"
            last = f"http {r.status_code}"
        except Exception as e:
            last = str(e)
        time.sleep(1.5 * (attempt + 1))
    return url, "error: " + last


verify_failed = []
with ThreadPoolExecutor(VERIFY_THREADS) as ex:
    futures = [ex.submit(head, j) for j in jobs]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="verify"):
        url, status = fut.result()
        if status != "ok":
            verify_failed.append((url, status))

report = {
    "edition": EDITION_IDENTIFIER,
    "bitrates": BITRATES,
    "path_pattern": f"{CDN_BASE}/{{bitrate}}/{EDITION_IDENTIFIER}/{{globalWordNumber}}.mp3",
    "words_total": 77432,
    "words_with_audio": len(TARGETS),
    "words_without_source_audio": sorted(MISSING_AUDIO),
    "files_expected": len(jobs),
    "files_verified_ok": len(jobs) - len(verify_failed),
    "files_failed": verify_failed,
}
json.dump(report, open(os.path.join(WORK_DIR, "wordbyword_report.json"), "w"), indent=2)
print(json.dumps({k: v for k, v in report.items() if k != "files_failed"}, indent=2))
print("failed:", verify_failed[:20])
assert not verify_failed, "Some files are not reachable on the CDN — re-run Step 3, then this cell"
print("\nAll files verified. Next: purge the edition:* cache tags on the API.")
